In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator

import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing

In [2]:
spark = SparkSession.builder \
    .appName("Random_Forest_House_Price") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/14 22:42:19 WARN Utils: Your hostname, Keshabs-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.92 instead (on interface en0)
26/06/14 22:42:19 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/14 22:42:19 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/14 22:42:20 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/06/14 22:42:20 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


In [3]:
housing = fetch_california_housing()

df_pandas = pd.DataFrame(
    housing.data,
    columns=housing.feature_names
)

df_pandas["MedHouseVal"] = housing.target

df_pandas.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


In [4]:
df_pandas["RoomsPerHousehold"] = df_pandas["AveRooms"] / df_pandas["AveOccup"]
df_pandas["PopulationPerHousehold"] = df_pandas["Population"] / df_pandas["AveOccup"]

df_pandas.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal,RoomsPerHousehold,PopulationPerHousehold
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526,2.732919,126.0
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585,2.956685,1138.0
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521,2.957661,177.0
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413,2.283154,219.0
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422,2.879646,259.0


In [5]:
df_pandas["RoomsPerHousehold"] = df_pandas["AveRooms"] / df_pandas["AveOccup"]
df_pandas["PopulationPerHousehold"] = df_pandas["Population"] / df_pandas["AveOccup"]

df_pandas.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal,RoomsPerHousehold,PopulationPerHousehold
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526,2.732919,126.0
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585,2.956685,1138.0
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521,2.957661,177.0
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413,2.283154,219.0
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422,2.879646,259.0


In [6]:
df = spark.createDataFrame(df_pandas)

df.show(5)
df.printSchema()

[Stage 0:>                                                          (0 + 1) / 1]

+------+--------+------------------+------------------+----------+------------------+--------+---------+-----------+------------------+----------------------+
|MedInc|HouseAge|          AveRooms|         AveBedrms|Population|          AveOccup|Latitude|Longitude|MedHouseVal| RoomsPerHousehold|PopulationPerHousehold|
+------+--------+------------------+------------------+----------+------------------+--------+---------+-----------+------------------+----------------------+
|8.3252|    41.0| 6.984126984126984|1.0238095238095237|     322.0|2.5555555555555554|   37.88|  -122.23|      4.526|2.7329192546583854|    126.00000000000001|
|8.3014|    21.0| 6.238137082601054|0.9718804920913884|    2401.0| 2.109841827768014|   37.86|  -122.22|      3.585|2.9566847147022073|                1138.0|
|7.2574|    52.0| 8.288135593220339| 1.073446327683616|     496.0|2.8022598870056497|   37.85|  -122.24|      3.521| 2.957661290322581|                 177.0|
|5.6431|    52.0|5.8173515981735155|1.07305936

Traceback (most recent call last):
  File "/opt/homebrew/opt/apache-spark/libexec/python/lib/pyspark.zip/pyspark/daemon.py", line 233, in manager
    code = worker(sock, authenticated)
  File "/opt/homebrew/opt/apache-spark/libexec/python/lib/pyspark.zip/pyspark/daemon.py", line 87, in worker
    outfile.flush()
    ~~~~~~~~~~~~~^^
BrokenPipeError: [Errno 32] Broken pipe
                                                                                

In [7]:
target_col = "MedHouseVal"

feature_cols = [
    "MedInc",
    "HouseAge",
    "AveRooms",
    "AveBedrms",
    "Population",
    "AveOccup",
    "Latitude",
    "Longitude",
    "RoomsPerHousehold",
    "PopulationPerHousehold"
]

print("Target column:", target_col)
print("Number of features:", len(feature_cols))
print("Feature columns:", feature_cols)

Target column: MedHouseVal
Number of features: 10
Feature columns: ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude', 'RoomsPerHousehold', 'PopulationPerHousehold']


In [8]:
assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

df_model = assembler.transform(df)

df_model.select("features", target_col).show(5, truncate=False)

+-------------------------------------------------------------------------------------------------------------------------------+-----------+
|features                                                                                                                       |MedHouseVal|
+-------------------------------------------------------------------------------------------------------------------------------+-----------+
|[8.3252,41.0,6.984126984126984,1.0238095238095237,322.0,2.5555555555555554,37.88,-122.23,2.7329192546583854,126.00000000000001]|4.526      |
|[8.3014,21.0,6.238137082601054,0.9718804920913884,2401.0,2.109841827768014,37.86,-122.22,2.9566847147022073,1138.0]            |3.585      |
|[7.2574,52.0,8.288135593220339,1.073446327683616,496.0,2.8022598870056497,37.85,-122.24,2.957661290322581,177.0]               |3.521      |
|[5.6431,52.0,5.8173515981735155,1.0730593607305936,558.0,2.547945205479452,37.85,-122.25,2.283154121863799,219.0]              |3.413      |
|[3.84

Traceback (most recent call last):
  File "/opt/homebrew/opt/apache-spark/libexec/python/lib/pyspark.zip/pyspark/daemon.py", line 233, in manager
    code = worker(sock, authenticated)
  File "/opt/homebrew/opt/apache-spark/libexec/python/lib/pyspark.zip/pyspark/daemon.py", line 87, in worker
    outfile.flush()
    ~~~~~~~~~~~~~^^
BrokenPipeError: [Errno 32] Broken pipe


In [9]:
train_df, test_df = df_model.randomSplit([0.8, 0.2], seed=42)

print("Training rows:", train_df.count())
print("Testing rows:", test_df.count())

Training rows: 16647
Testing rows: 3993


In [10]:
rf = RandomForestRegressor(
    featuresCol="features",
    labelCol=target_col,
    predictionCol="prediction",
    numTrees=100,
    maxDepth=5,
    seed=42
)

In [ ]:
rf_model = rf.fit(train_df)